# Custom State
Node(Agent)간에 공유되는 상태정보를 커스터마이징할수 있다.

**최신 장소정보 공유**
- tool 장소검색api: naver/kakao/google map api(키워드 검색기반)
- tool 자연어 -> 검색을 위한 키워드추출
    - 나 여기 독산동인데, 여기 괜찮은 까페 추천해줘 -> 독산동 까페

In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os


load_dotenv()


os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
KAKAO_API_KEY = os.getenv('KAKAO_API_KEY')

os.environ['LANGSMITH_ENDPOINT'] = (
    'https://api.smith.langchain.com'
)

os.environ['LANGSMITH_PROJECT'] = (
    'skn34-langchain'
)

os.environ['LANGSMITH_API_KEY'] = (
    os.getenv('LANGSMITH_API_KEY')
)

os.environ['LANGSMITH_TRACING'] = 'true'

## 카카오 장소검색 도구

https://developers.kakao.com/docs/latest/ko/local/dev-guide#search-by-keyword

In [2]:
from langchain.tools import tool
import requests
from pprint import pprint
from typing import List, TypedDict


class kakaoplace(TypedDict):
    name: str
    address: str
    url: str


# 키워드를 이용해 카카오 장소 검색 수행
@tool
def kakao_place_search(query: str) -> List[kakaoplace]:
    """
    카카오 장소 검색 API를 호출하기 위한 도구

    Args:
        query: 검색어(키워드 기반)

    Returns:
        검색 결과
    """

    url = 'https://dapi.kakao.com/v2/local/search/keyword.json'

    headers = {
        'Authorization': f'KakaoAK {KAKAO_API_KEY}'
    }

    params = {
        'query': query,
        'size': 5
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    data = response.json()
    data = data['documents']

    return [
        {
            'name': place['place_name'],
            'address': place['address_name'],
            'url': place['place_url']
        }
        for place in data
    ]


pprint(
    kakao_place_search.invoke({
        'query': '독산동 맛집'
    })
)

[{'address': '서울 금천구 독산동 163-2',
  'name': '왕래성',
  'url': 'http://place.map.kakao.com/538642507'},
 {'address': '서울 금천구 독산동 179-8',
  'name': '청정회센터',
  'url': 'http://place.map.kakao.com/9965001'},
 {'address': '서울 금천구 독산동 288-9',
  'name': '불맛전문관사부간짬뽕 독산직영점',
  'url': 'http://place.map.kakao.com/962610671'},
 {'address': '서울 금천구 독산동 1113',
  'name': '소하떡집',
  'url': 'http://place.map.kakao.com/1448413740'},
 {'address': '서울 금천구 독산동 378-428',
  'name': '돈유창',
  'url': 'http://place.map.kakao.com/1002142708'}]


### 키워드 추출 도구

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool


llm = init_chat_model(
    'gpt-5.4-mini'
)


@tool
def extract_keyword(user_request: str) -> str:
    """
    사용자 요청에서 장소 검색 API를 사용하기 위한
    짧고 명확한 키워드를 추출하는 도구입니다.

    장소 검색 도구(kakao_place_search)를 사용하기 전에
    사용자의 요청을 먼저 변환하세요.
    """

    prompt = PromptTemplate.from_template('''
당신은 사용자의 요청을 장소 검색에 필요한 키워드로
변환하는 전문가입니다.

장문의 사용자 요청에서 키워드를 한 문장으로 추출해 주세요.

- 키워드는 띄어쓰기로 구분된 한 문장이어야 합니다.
- 다른 설명은 하지 말고 키워드만 추출합니다.

[사용자 요청]
{user_request}

[예시]
- 여기 독산동인데, 소문난 맛집 좀 알려줘
  -> 독산동 맛집

- 부산 핫플 좀 알려줘
  -> 부산 맛집

[금지어]
- 소문난
- 핫플
''')

    chain = prompt | llm

    response = chain.invoke({
        'user_request': user_request
    })

    return (
        response.content
        .strip()
        .strip('"')
        .strip("'")
    )


print(
    extract_keyword.invoke(
        '여기 독산역 근처인데, 소문난 맛집 좀 알려줘'
    )
)

print(
    extract_keyword.invoke(
        '여기 서울역 근처인데, 소문난 맛집 좀 알려줘'
    )
)

print(
    extract_keyword.invoke(
        '여기 일산역 근처인데, 소문난 맛집 좀 알려줘'
    )
)

독산역 맛집
서울역 맛집
일산역 맛집


### 그래프 구성
- 키워드 추출 -> 카카오 장소 검색 -> 상태(State)에 추출

In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, START
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver


load_dotenv()


# LLM 생성
llm = init_chat_model(
    'gpt-5.4-mini'
)


# Tavily 검색 도구 생성
tavily_tool = TavilySearch(
    max_results=3
)


# 사용할 도구 목록
tools = [
    tavily_tool
]


# LLM에 도구 연결
llm_with_tools = llm.bind_tools(tools)


# 챗봇 노드 함수
def chatbot(state: MessagesState):
    response = llm_with_tools.invoke(
        state['messages']
    )

    return {
        'messages': [response]
    }


# 그래프 생성
builder = StateGraph(MessagesState)

tool_node = ToolNode(tools)


# 노드 등록
builder.add_node(
    'chatbot',
    chatbot
)

builder.add_node(
    'tools',
    tool_node
)


# 그래프 연결
builder.add_edge(
    START,
    'chatbot'
)

builder.add_conditional_edges(
    'chatbot',
    tools_condition
)

builder.add_edge(
    'tools',
    'chatbot'
)


# 메모리와 그래프 생성
memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

print('builder 및 graph 생성 완료!')

builder 및 graph 생성 완료!


그래프 실행

In [6]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[
        list[BaseMessage],
        add_messages
    ]

    search_query: str | None
    search_results: list[dict] | None

In [7]:
initial_state: State = {
    'messages': [
        (
            'human',
            '독산역 맛집 알려줘'
        )
    ],
    'search_query': None,
    'search_results': None
}


config = {
    'configurable': {
        'thread_id': 'user1'
    }
}


final_state = graph.invoke(
    initial_state,
    config=config
)

final_state

{'messages': [HumanMessage(content='독산역 맛집 알려줘', additional_kwargs={}, response_metadata={}, id='3568016e-a4ea-4a2d-be85-2e68a9c0f7a7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 1275, 'total_tokens': 1313, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EJwD8nq1vftzS8jTPAUHCvAUXUPIO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0662e-3fc6-7621-91e9-e4a42f923c7b-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '독산역 맛집 추천', 'search_depth': 'advanced', 'topic': 'general', 'includ

In [8]:
from IPython.display import Markdown # 

Markdown(final_state['messages'][-1].content) # 마지막 메시지 출력

독산역 맛집 몇 군데 추천드릴게요. 취향별로 골라보면 좋아요.

### 가성비/점심
- **도원 중식뷔페**  
  중식뷔페로 가성비가 좋다는 평이 많아요. 짜장면, 탕수육, 짬뽕, 잡채 등 종류가 다양해서 든든하게 먹기 좋습니다.

- **서대문김치찜**  
  돼지김치찜, 고등어김치찜이 대표 메뉴예요. 점심에 밥 한 끼 든든하게 먹기 좋습니다.

### 국밥/해장
- **실비순댓국**  
  오래된 내공의 내장탕/순댓국집으로 알려져 있어요. 해장이나 뜨끈한 국물 생각날 때 좋아요.

### 면/돈가스
- **히나타 자가제면**  
  자가제면 우동과 수제 돈가스가 유명해요. 점심이나 저녁으로 무난하고, 혼밥하기도 괜찮습니다.

### 특별한 메뉴
- **라티시크릿셰프**  
  프랑스식 요리/파스타/코스 느낌이라 데이트나 기념일에 어울려요.
- **뉴욕스타일보일링크랩**  
  해산물, 특히 보일링크랩 같은 색다른 메뉴를 먹고 싶을 때 추천해요.

원하시면 제가  
**1) 점심용**, **2) 저녁/회식용**, **3) 혼밥용**, **4) 술 한잔하기 좋은 곳**  
이렇게 나눠서 독산역 맛집을 더 골라드릴게요.

In [12]:
# LangGraph 스트리밍 실행:
# 노드별 실행 결과를 실시간 확인

initial_state: State = {
    'messages': [
        (
            'human',
            '독산역 돈까스 맛집 알려줘'
        )
    ],
    'search_query': None,
    'search_results': None
}


user1_config = {
    'configurable': {
        'thread_id': 'user1'
    }
}


for chunk in graph.stream(
    initial_state,
    config=user1_config,
    stream_mode='updates'
):
    for node, update in chunk.items():
        print(f'{node} node')

        # 해당 노드가 messages를 반환한 경우에만 출력
        if 'messages' in update and update['messages']:
            update['messages'][-1].pretty_print()

        # 나머지 상태도 확인
        else:
            print(update)

        print()

chatbot node
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_UzrJKnjt5VaROkuq7MHyU6g1)
 Call ID: call_UzrJKnjt5VaROkuq7MHyU6g1
  Args:
    query: 독산역 돈까스 맛집
    search_depth: advanced
    topic: general
    include_images: False

tools node
================================= Tool Message =================================
Name: tavily_search

{"query": "독산역 돈까스 맛집", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://blog.naver.com/PostView.nhn?blogId=kio4545&logNo=224348949996&redirect=Dlog&widgetTypeCall=true", "title": "독산역 돈까스 맛집 히나타 자가제면 : 네이버 블로그", "content": "독산역 돈까스 맛집 히나타 자가제면\n\n프로파일\n\n서울특별시 금천구 가산디지털1로 16 107-1R호\n\n안녕하세요. 오늘은 독산역 우동, 소바, 돈까스 맛집 히나타에 다녀왔습니다.\n\n가산디지털단지와 독산역에는 직장인들이 많아 점심이나 저녁 먹을 곳을 많이들 찾으시는데요.\n\n돈까스 하면 남녀노소 누구나 좋아하잖아요?\n\n그래서 호불호가 없는 맛집 히나타를 소개해드리고자 합니다.\n\n독산역 돈까스 맛집 히나타는 가산디지털1로 16 SKV1타워 에이피타워 1층에 위치해있습니다.\n\n독산역 2번출구에서 도보 5분거리에 있으며, 건물내 주차장에 주차도 가능합니다.\